# 04 — Viz (exploration)

**billboard-hot100.** Explore how music **genres** rose and fell at the top of the
Billboard Hot 100 over the decades. The framing is now **LOCKED (2026-10-15)** — see
the ⭐ banner below; the lead **bump chart** (and the full social set) is built in
`06-viz-social`. This notebook is the exploratory record that led to those decisions,
updated to reflect the final story (incl. the song-level pre/post comparison in §5).

> ⭐ This notebook **CONSUMES** the chart-ready tables `03-prepare` built
> (`chart_genre_by_year`, `chart_genre_by_decade`, `chart_genre_rank_top`,
> `chart_genre_number_ones_by_decade`). It opens DuckDB **read-only** and does
> **not** create any `chart_*` table. If exploration shows we need a new cut, the
> fix is to amend `03-prepare` and re-run — never to build chart data here.

**On the bump template:** the shared kit has no `bump` type yet — building it is
*this project's* capability add, scheduled for `06-viz-social`. A bump is just a
specialised line of **rank vs year**, so for exploration we draw the rank race with
the existing shared **`line`** template: we plot an **inverted** rank (so rank 1
sits at the TOP) by charting `(N+1 − rank)` and relabelling the y ticks back to
`#rank`. The goal is to let the owner SEE and choose the framing, not to produce
the final bump.

The venv is Python 3.14 (matplotlib RecursionErrors), so every chart is rendered
with the shared **Pillow factory** (`render_chart`) and displays **inline**.

In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched.
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / '..' / '..' / 'shared'))

import duckdb, pandas as pd
pd.set_option('display.max_rows', 120)

from src.ingest import load_config
from colors import c, CATEGORICAL_10
from chart_factory import render_chart

cfg = load_config('config.yaml')
# READ-ONLY: 04-viz only consumes 03's prepared tables.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
print('tables:', [r[0] for r in con.execute('SHOW TABLES').fetchall()])

## Rank-bump helper (via the `line` template)

A tiny helper that turns a long `(year, genre, rank)` frame into the wide,
inverted-rank form the `line` template wants, then renders it. We pass the
line template a `y_fmt` lambda so the axis still reads as `#1 … #N` even though
the plotted value is `(N+1 − rank)`. One **genre = one line**, labelled at its
last point. This is a stand-in for the real bump template coming in `06`.

In [ ]:
def bump_from_long(df_long, rank_col='rank', x_col='year', genre_col='genre',
                   genres=None, palette=None, title='', subtitle='', source=None,
                   filename=None, x_tick_step=10):
    """Render a rank bump (rank 1 at top) from a long (x, genre, rank) frame
    using the shared `line` template. Returns nothing; displays inline."""
    d = df_long.copy()
    if genres is None:
        genres = list(d[genre_col].unique())
    d = d[d[genre_col].isin(genres)]
    N = int(d[rank_col].max())
    # Invert: rank 1 -> top. Plot (N+1 - rank).
    d['_inv'] = (N + 1) - d[rank_col]
    wide = d.pivot_table(index=x_col, columns=genre_col, values='_inv',
                         aggfunc='mean').reset_index()
    if palette is None:
        palette = CATEGORICAL_10
    # Order series by their most-recent rank so the legend/labels read sensibly.
    last = d.sort_values(x_col).groupby(genre_col)[rank_col].last().sort_values()
    ordered = [g for g in last.index if g in wide.columns]
    series = [{'col': g, 'label': g, 'color': palette[i % len(palette)]}
              for i, g in enumerate(ordered)]
    render_chart({
        'type': 'line',
        'table': wide,
        'x_col': x_col,
        'series': series,
        'y_fmt': (lambda v, _N=N: f'#{int(_N + 1 - round(v))}'),
        'x_axis_label': '',  # year/decade ticks are self-evident (project standard)
        'y_axis_label': 'Rank among set (1 = most chart appearances)',
        'y_min': 0.5, 'y_max': N + 0.5,
        'markers': True, 'label_last': True, 'legend': False,
        'x_tick_step': x_tick_step,
        'title': title, 'subtitle': subtitle, 'source': source,
        'filename': filename,
    })

SRC = 'Billboard Hot 100 (chart history) · genre via MusicBrainz · @unwelcomedata'
print('helper ready')

## ⭐ FINAL STORY — LOCKED 2026-10-15 (owner)

The framing below is **settled**; the cells in this notebook are the record of what
was explored, but the decisions are no longer open. The owner picked:

1. **LEAD = annual genre rank BUMP** — grain = **year** (owner: *"I prefer the bump
   chart by year, not decade"*), **top-6 real genres**, **start 1970** (*"we don't
   need pre-1970"*), **Other/Unknown EXCLUDED** (*"we don't need the chart with
   other"*). Rank re-densed within the top-6 for 1970+. Rendered for real in
   `06-viz-social` with the kit's **new `bump` template**.
2. **SUPPORT = genre-share stacked** — the 100%-stacked-by-decade part-of-whole
   (owner: *"Like the genre share stacked chart"*). All 12 genres incl. Other/Unknown
   (grey) so it's a faithful part-of-whole.
3. **Metal cut KEPT, but NOT as a social chart** — it is the **default view of the
   genre dropdown picker** only (owner: *"the metal chart makes no sense by itself,
   that is just for the drop down picker"*). **No other `Other` genre is broken out**
   (owner asked; answer = NO — only Reggae recurs and it lacks tag volume).
4. **Two NEW song-level social charts** — pre-streaming vs streaming-era
   longest-charting songs (see §5 below), split at the 2013 streaming break.
5. **Genre selection = a Pages DROPDOWN PICKER, not social** (`docs/genres.html`,
   built by `scripts/build_genre_pages.py`; all 12 genres, defaults to Metal).
6. **New x-axis standard:** no "Year" title when the ticks are obviously years
   (pass `x_axis_label=""`). Applied throughout — including §2b below, which the
   owner specifically flagged (*"get rid of it on chart 2"*).
7. **2013 streaming series break** annotated on the year-axis charts (dashed
   "2013: streaming added" marker) and caveated on the categorical/song charts.

---
## 1. `Other/Unknown` handling — why the lead EXCLUDES it (settled)

`Other/Unknown` is **not a genre** — it is the coverage residual (unmatched
artists + the long tail below the ≥ 20-appearance enrichment threshold). But it is
big: the `03` sanity check showed it ranks **#1 in the 1950s, 1960s and 1990s**.
That is exactly why the LEAD excludes it and starts at 1970. We keep all three
exploratory cuts below as the record of how that call was reached:

- **(a) top-6 real genres only** — clean 6-line race from `chart_genre_rank_top`.
- **(b) include `Other/Unknown`** — 7 lines; shows how 'unknown' the early decades
  are, but a dominant grey not-a-genre line crowds the top. Rejected for the lead.
- **(c) top-6 real, 1970-start** — ✅ **THIS is the locked lead framing**: honest
  AND clean, it just doesn't claim to rank genres in the thin-coverage 1958–69 era.

### (a) Top-6 real genres only — the clean race (`chart_genre_rank_top`)

In [ ]:
top6 = con.execute('SELECT * FROM chart_genre_rank_top ORDER BY year, rank').df()
bump_from_long(
    top6,
    title='Genre rank on the Billboard Hot 100, by year',
    subtitle='Top 6 genres by chart appearances; Other/Unknown excluded. Coverage is thin pre-1970.',
    source=SRC, filename='explore_bump_top6_year',
)

### (b) Include `Other/Unknown` as its own line (7 lines, re-ranked within the set)
Take the 6 real genres **plus** `Other/Unknown` from `chart_genre_by_year`, then
re-rank *within those seven* each year (so it's a dense 1..7 race). The grey line
is `Other/Unknown` — watch it dominate the early decades and fade as coverage
improves.

In [ ]:
seven = con.execute('''
    WITH keep AS (
        SELECT * FROM chart_genre_by_year
        WHERE genre IN (SELECT DISTINCT genre FROM chart_genre_rank_top)
           OR genre = 'Other/Unknown'
    )
    SELECT year, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY year ORDER BY appearances DESC) AS rank
    FROM keep
    ORDER BY year, rank
''').df()
# Palette: give Other/Unknown a flat grey so it reads as 'not a genre'.
genres7 = seven.sort_values('year').groupby('genre')['rank'].last().sort_values().index.tolist()
pal7 = []
ci = 0
for g in genres7:
    if g == 'Other/Unknown':
        pal7.append(c('gray'))
    else:
        pal7.append(CATEGORICAL_10[ci % len(CATEGORICAL_10)]); ci += 1
bump_from_long(
    seven, genres=genres7, palette=pal7,
    title='Genre rank on the Billboard Hot 100, by year (with Other/Unknown)',
    subtitle='Top 6 genres + Other/Unknown (grey), re-ranked within the set. Other/Unknown is a coverage residual, not a genre.',
    source=SRC, filename='explore_bump_seven_year',
)

### (c) Top-6 real, **1970-start** — avoid the thin-coverage era
Same six lines as (a), but only from 1970 on. Ranks are re-densed within the
top-6 for the 1970+ window so the race reads cleanly where coverage is strong.

In [ ]:
top6_1970 = con.execute('''
    SELECT year, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY year ORDER BY appearances DESC) AS rank
    FROM chart_genre_rank_top
    WHERE year >= 1970
    ORDER BY year, rank
''').df()
bump_from_long(
    top6_1970,
    title='Genre rank on the Billboard Hot 100, 1970–present',
    subtitle='Top 6 genres by chart appearances, from 1970 (when genre coverage is strong). Other/Unknown excluded.',
    source=SRC, filename='explore_bump_top6_1970',
)

**Outcome (LOCKED).** (a) hides how little we know about the 1958–69 era; (b) puts
a fat grey not-a-genre line on top for two decades (weak lead, invites 'what is
Other?'); **(c) was chosen** — top-6 real genres, **1970-start**, Other excluded —
the honest-and-clean cut that simply doesn't rank genres where coverage is too thin.
This is the lead bump in `06-viz-social` (rendered with the new `bump` template).

## 2. Count/rank vs share

The bump ranks genres by **appearances** (count). A rank-bump answers *who's on
top*; it hides *by how much*. The alternative framing is **share of the Hot 100** —
a line chart with share (%) on the y-axis — which answers *how much of the chart
each genre actually held*. The two tell slightly different stories (a genre can be
#1 by a hair, or #2 by a mile), so we show both.

### 2a. Confirm the rank bump matches the `rank` column
Sanity check: the helper re-pivots from the `rank` column `03` already computed, so
the top-6 bump IS the stored rank. Spot-check a couple of years.

In [ ]:
chk = con.execute('''
    SELECT year, genre, appearances, rank
    FROM chart_genre_rank_top
    WHERE year IN (1975, 1995, 2020)
    ORDER BY year, rank
''').df()
display(chk)
print('rank is a direct read of the 03 `rank` column — the bump plots it, does not recompute it.')

### 2b. Share-of-Hot-100 lines (share %, not rank) — the 'how much' framing
Same six genres, but y = **share of that year's chart slots**. Labels at the last
point. This shows magnitude (e.g. the rise of Hip-Hop/Rap's actual share) that a
rank bump flattens.

In [ ]:
share6 = con.execute('SELECT * FROM chart_genre_rank_top ORDER BY year').df()
genres6 = share6.sort_values('year').groupby('genre')['share'].last().sort_values(ascending=False).index.tolist()
wide_share = share6.pivot_table(index='year', columns='genre', values='share', aggfunc='mean').reset_index()
series_share = [{'col': g, 'label': g, 'color': CATEGORICAL_10[i % len(CATEGORICAL_10)]}
                for i, g in enumerate(genres6)]
render_chart({
    'type': 'line',
    'table': wide_share,
    'x_col': 'year',
    'series': series_share,
    'x_axis_label': '',
    'y_axis_label': 'Share of Hot 100 chart slots (%)',
    'y_min': 0,
    'markers': False, 'label_last': True, 'legend': False, 'x_tick_step': 10,
    'title': 'Genre share of the Billboard Hot 100, by year',
    'subtitle': 'Percent of each year\'s chart slots held by the top 6 genres. Coverage thin pre-1970.',
    'source': SRC,
    'filename': 'explore_share_top6_year',
})

**Outcome (LOCKED).** Lead with the **rank bump**; the **genre-share stacked**
chart is the supporting 'how much' part-of-whole (owner likes it). The share-LINES
explored here informed that call but the SUPPORT chart that ships is the
100%-stacked-by-decade view (§4b), not these lines.

## 3. Year grain vs decade grain

The annual bump (69 years) shows the real crossing *years* but is noisier; the
decade bump (`chart_genre_by_decade`, restricted to the same top-6 and re-ranked)
is a smooth arc that reads cleanly on a phone but loses the exact hand-off year.
Show both.

### 3a. Annual bump (same as 1a, repeated here for the grain comparison)

In [ ]:
bump_from_long(
    top6,
    title='Genre rank on the Billboard Hot 100, by year (annual grain)',
    subtitle='Top 6 genres; annual rank shows the exact crossing years. Coverage thin pre-1970.',
    source=SRC, filename='explore_bump_top6_year_grain', x_tick_step=10,
)

### 3b. Decade bump — smoother arc for social

In [ ]:
dec6 = con.execute('''
    WITH keep AS (
        SELECT * FROM chart_genre_by_decade
        WHERE genre IN (SELECT DISTINCT genre FROM chart_genre_rank_top)
    )
    SELECT decade, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY decade ORDER BY appearances DESC) AS rank
    FROM keep
    ORDER BY decade, rank
''').df()
bump_from_long(
    dec6, x_col='decade',
    title='Genre rank on the Billboard Hot 100, by decade',
    subtitle='Top 6 genres by chart appearances, re-ranked within the set per decade.',
    source=SRC, filename='explore_bump_top6_decade', x_tick_step=10,
)

**Outcome (LOCKED).** The owner chose **annual grain** (*"I prefer the bump chart
by year, not decade"*). Annual shows the genuine crossing years — the whole appeal
of a bump — at the cost of some wiggle; decade was NOT chosen. The decade cut here
stays only as the exploratory record.

## 4. Supporting cuts

Two cheap context charts the owner asked for.

### 4a. Metal's trajectory (owner's personal interest) → PICKER DEFAULT, not social
Metal is **not** in the top-6, so it won't appear in the main bump. The owner wants
to SEE where it lands — peak **276 appearances in 2001** — but decided the Metal
chart **makes no sense as a standalone social post** (*"that is just for the drop
down picker"*). So Metal is **not a social chart**: it is the **default view of the
genre dropdown picker** (`docs/genres.html`, built by `scripts/build_genre_pages.py`,
which renders this same appearances-per-year line for all 12 genres). The cell below
stays as the exploratory record / the shape the picker shows.

In [ ]:
metal = con.execute('''
    SELECT year, appearances, share, rank
    FROM chart_genre_by_year WHERE genre = 'Metal' ORDER BY year
''').df()
peak = metal.loc[metal['appearances'].idxmax()]
print(f"Metal peak: {int(peak['appearances'])} appearances in {int(peak['year'])} "
      f"(rank #{int(peak['rank'])} that year, {peak['share']}% of the chart)")
render_chart({
    'type': 'line',
    'table': metal,
    'x_col': 'year',
    'series': [{'col': 'appearances', 'label': 'Metal', 'color': c('brown_red')}],
    'x_axis_label': '',
    'y_axis_label': 'Hot 100 chart appearances',
    'y_min': 0, 'markers': True, 'label_last': False, 'legend': False,
    'value_labels': False, 'x_tick_step': 10,
    'title': 'Metal on the Billboard Hot 100, by year',
    'subtitle': 'Chart appearances per year. Metal sits just outside the top 6 (peak 276 in 2001).',
    'source': SRC,
    'filename': 'explore_metal_year',
})

### 4b. Genre share by decade — 100% stacked
A part-of-whole view: each decade's chart slots split across genres. Uses the
shared `stacked_100pct_bars` template on `chart_genre_by_decade` (all 12 genres,
so `Other/Unknown` is visible as its own band — honest about coverage per decade).

In [ ]:
stacked = con.execute('''
    SELECT decade, genre, share
    FROM chart_genre_by_decade ORDER BY decade, genre
''').df()
wide_stacked = stacked.pivot_table(index='decade', columns='genre', values='share',
                                   aggfunc='sum').fillna(0).reset_index()
# group labels must be strings for the stacked template (e.g. '1950s').
wide_stacked['decade'] = wide_stacked['decade'].astype(int).astype(str) + 's'
genre_order = ['Rock', 'R&B/Soul', 'Pop', 'Country', 'Hip-Hop/Rap', 'Dance/Electronic',
               'Latin', 'Folk', 'Jazz', 'Blues', 'Metal', 'Other/Unknown']
genre_order = [g for g in genre_order if g in wide_stacked.columns]
segments = [
    {'col': g, 'label': g,
     'color': (c('gray') if g == 'Other/Unknown' else CATEGORICAL_10[i % len(CATEGORICAL_10)])}
    for i, g in enumerate(genre_order)
]
try:
    render_chart({
        'type': 'stacked_100pct_bars',
        'table': wide_stacked,
        'group_col': 'decade',
        'segments': segments,
        'show_legend': True,
        'title': 'Genre share of the Billboard Hot 100, by decade',
        'subtitle': 'Each decade\'s chart slots split across genres (Other/Unknown in grey). Coverage thin in the 1950s–60s.',
        'source': SRC,
        'filename': 'explore_genre_share_stacked_decade',
    })
except Exception as e:
    # The stacked template signature can vary; if it doesn't take this shape cheaply,
    # fall back to a plain inline table rather than over-building (03 owns chart data).
    print('stacked_100pct_bars did not take this shape cleanly:', e)
    print('Falling back to the numeric share-by-decade matrix (explore via the share-lines above):')
    display(wide_stacked.round(1))

**Note.** This is the SUPPORT chart that ships (owner likes the genre-share
stacked). Kept as all 12 genres incl. Other/Unknown (grey) so it's a faithful
part-of-whole; the 2010s/20s decades reflect the post-2013 streaming formula
(caveat in the social subtitle).

## 5. Song-level — longest-charting songs, pre-streaming vs streaming era

Added to the FINAL social set (2026-10-15): two song-grain charts contrasting the
**longest-charting Hot 100 songs** before and after the **2013 streaming break**.
`03-prepare` owns the shaped tables (`chart_songs_pre_streaming` /
`chart_songs_streaming_era` — top-10 songs by total weeks on chart, split on first
chart year 1958–2012 vs 2013–2026). `04` CONSUMES them to confirm the story; `06`
renders them (ranked bars, song = label, artist = sublabel, bar color = genre,
shared genre palette across both). Explored inline here so this notebook reflects
what `06` ships.

**The story (verified below):** streaming-era songs run **far longer** (top ~112
weeks vs ~87 pre-streaming) and the **genre mix flips** from Pop/Rock/Country to
R&B-Soul / Dance-Electronic / Pop (no Rock in the streaming top-10).

**Caveat (kept, not hardened):** the streaming ramp bled into the late-2000s/early-
2010s, so the pre-streaming top is dominated by 2008–2012 songs plus perennial
Christmas songs — the split is at 2013 with a caveat line, not a hard wall.

In [ ]:
song_pre = con.execute('SELECT song, artist, genre, weeks FROM chart_songs_pre_streaming ORDER BY weeks DESC, song').df()
song_stream = con.execute('SELECT song, artist, genre, weeks FROM chart_songs_streaming_era ORDER BY weeks DESC, song').df()
print('PRE-STREAMING (first year 1958-2012) — top by weeks on chart:')
display(song_pre)
print('STREAMING ERA (first year 2013-2026) — top by weeks on chart:')
display(song_stream)
print(f"pre max weeks = {int(song_pre['weeks'].max())}  |  streaming max weeks = {int(song_stream['weeks'].max())}")
print('pre genres:   ', sorted(song_pre['genre'].unique()))
print('stream genres:', sorted(song_stream['genre'].unique()))

**Confirmed:** pre #1 = **Radioactive / Imagine Dragons (87 wks, Rock)**; streaming
#1 = **Lose Control / Teddy Swims (112 wks, R&B/Soul)**. The genre sets differ (Rock
present pre, absent streaming), confirming the mix flip. These are the two song
charts built in `06-viz-social`; the single tall STACKED comparison of the two is
the Pages/web object (not a social post).

## Status — FINAL story locked; `06-viz-social` BUILT

All framing is **LOCKED (2026-10-15)** — see the banner at the top. The social set
is **4 charts**: (1) annual genre rank **bump** (top-6, 1970+, no Other, 2013
streaming vline, new `bump` template), (2) genre-**share** 100%-stacked by decade,
(3) pre-streaming + (4) streaming-era longest-charting songs. **Metal** is the
**genre picker default**, not a social chart. Genre selection is a **Pages dropdown
picker** (`docs/genres.html`). All year-axis charts drop the "Year" title; the
**2013 streaming break** is annotated/caveated and documented in SOURCES.md +
`_sources`. `06-viz-social` and the picker are built; this notebook reflects the
shipped story.

---
## Cleanup
Close the read-only DuckDB connection (single-writer lock).

In [ ]:
con.close()
print('connection closed')